# 02 · Train → evaluate → export (extinguisher + blue_box)

Trains a YOLOv8n detector on `datasets/go2_det/data.yaml` (made by notebook 01), evaluates it on the **Go2 test split**, picks the `conf_thres` for the ROS node, and exports the weights for the Jetson.

Model size stays at **n**: the Orin Nano has to keep up with the camera at ~8 Hz, together with everything else running on the robot.

| section | what |
|---|---|
| 1 | environment + dataset check |
| 2 | baseline: the old `fire_extinguisher_best.pt` on the new test split |
| 3 | training (optionally a second run from COCO weights for comparison) |
| 4 | evaluation: official mAP, threshold sweep → `CONF_THRES`, recall by object size, error gallery |
| 5 | export: `.pt` into `models/`, ONNX sanity check, model card, Jetson TensorRT commands |

In [ ]:
import json, math, random, shutil, time
from datetime import date
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm
import ultralytics
from ultralytics import YOLO

## 0 · Configuration

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_YAML = ROOT / 'datasets' / 'go2_det' / 'data.yaml'
RUNS_DIR = ROOT / 'runs'
RUN_NAME = 'go2_ext_bluebox'

# Start from the old extinguisher model (already knows extinguishers; its 1-class head is re-initialised for 2 classes)
INIT_WEIGHTS = ROOT / 'models' / 'fire_extinguisher_best.pt'
COMPARE_WITH_COCO_INIT = False      # also train from 'yolov8n.pt' (COCO) and keep whichever is better on val
OLD_MODEL = ROOT / 'models' / 'fire_extinguisher_best.pt'   # baseline for section 2 (None to skip)

EPOCHS = 150
PATIENCE = 30          # early stop after this many epochs without val improvement
IMGSZ = 640            # the RealSense colour stream is 640x480; letterboxed to 640
BATCH = 32
WORKERS = 8
DEVICE = 0
SEED = 0
CACHE = 'ram'          # 'ram' | 'disk' | False

# Online augmentation (on top of the offline augmentation from notebook 01).
# hsv_h is kept small because colour (red vs blue) separates the classes.
TRAIN_HYP = dict(hsv_h=0.01, hsv_s=0.5, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.4,
                 fliplr=0.5, mosaic=1.0, close_mosaic=15, mixup=0.0)

IOU_MATCH = 0.5        # IoU for TP matching in the custom evaluation
SWEEP_CONF = 0.01      # predictions are collected at this confidence and thresholded afterwards

EXPORT_NAME = 'go2_ext_bluebox_best'      # -> models/<EXPORT_NAME>.pt
JETSON_ULTRALYTICS = '8.4.137'            # version the old model was trained with; pin the same in Dockerfile.subscriber

random.seed(SEED); np.random.seed(SEED)

## 1 · Environment and dataset check

In [ ]:
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| CUDA', torch.version.cuda)
assert torch.cuda.is_available(), 'no CUDA GPU visible to torch'
print('GPU:', torch.cuda.get_device_name(DEVICE))
if ultralytics.__version__ != JETSON_ULTRALYTICS:
    print(f'WARNING: training with ultralytics {ultralytics.__version__}, Jetson pin is {JETSON_ULTRALYTICS}. '
          'Checkpoints usually load across close versions, but keep both ends pinned to the same version.')

cfg = yaml.safe_load(DATA_YAML.read_text())
DS = Path(cfg['path'])
names = cfg['names']
CLASSES = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
print('classes:', CLASSES)

def read_yolo(txt):
    txt = Path(txt)
    if not txt.exists():
        return np.zeros((0, 5), np.float32)
    rows = [l.split()[:5] for l in txt.read_text().splitlines() if len(l.split()) >= 5]
    return np.array(rows, np.float32).reshape(-1, 5)

def split_files(split):
    return sorted(p for p in (DS / cfg[split]).iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})

def label_path(img_p):
    parts = list(img_p.parts)
    parts[len(parts) - 1 - parts[::-1].index('images')] = 'labels'
    return Path(*parts).with_suffix('.txt')

rows = []
for s in ['train', 'val', 'test']:
    files = split_files(s)
    labs = [read_yolo(label_path(p)) for p in files]
    rows.append(dict(split=s, images=len(files), background=sum(len(l) == 0 for l in labs),
                     **{c: int(sum((l[:, 0] == i).sum() for l in labs)) for i, c in enumerate(CLASSES)}))
display(pd.DataFrame(rows).set_index('split'))

### Evaluation helpers

`model.val()` gives the official COCO-style mAP. The helpers below also collect raw predictions once, which lets us sweep the confidence threshold the ROS node will use, compute recall per object size, and draw the errors.

In [ ]:
def xywhn_to_xyxy(l, W, H):
    cx, cy, w, h = l[:, 1] * W, l[:, 2] * H, l[:, 3] * W, l[:, 4] * H
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)

def iou(a, b):
    if not len(a) or not len(b):
        return np.zeros((len(a), len(b)))
    tl, br = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(-1)
    area = lambda x: (x[:, 2:] - x[:, :2]).prod(-1)
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

def collect(model, split='test', class_map=None):
    """Run the model on a split. class_map: model class id -> our class id (default: by name).
    Returns a list of dicts: file, gt (cls, xyxy), pred (cls, conf, xyxy)."""
    if class_map is None:
        class_map = {i: CLASSES.index(n) for i, n in model.names.items() if n in CLASSES}
    files = split_files(split)
    out = []
    for res in tqdm(model.predict([str(p) for p in files], conf=SWEEP_CONF, imgsz=IMGSZ, stream=True,
                                  verbose=False, device=DEVICE), total=len(files), desc=f'predict {split}'):
        H, W = res.orig_shape
        g = read_yolo(label_path(Path(res.path)))
        c = res.boxes.cls.int().cpu().numpy()
        keep = np.array([k in class_map for k in c], bool)
        out.append(dict(file=res.path, W=W, H=H,
                        gt_cls=g[:, 0].astype(int), gt_xyxy=xywhn_to_xyxy(g, W, H),
                        p_cls=np.array([class_map[k] for k in c[keep]], int),
                        p_conf=res.boxes.conf.cpu().numpy()[keep],
                        p_xyxy=res.boxes.xyxy.cpu().numpy()[keep]))
    return out

def match_image(r, cls, thr):
    """Greedy matching for one class at confidence >= thr. Returns (tp flags of preds, matched flags of gts)."""
    pm = (r['p_cls'] == cls) & (r['p_conf'] >= thr)
    gm = r['gt_cls'] == cls
    pb, pc, gb = r['p_xyxy'][pm], r['p_conf'][pm], r['gt_xyxy'][gm]
    order = np.argsort(-pc)
    ious = iou(pb[order], gb)
    tp, used = np.zeros(len(pb), bool), np.zeros(len(gb), bool)
    for k in range(len(pb)):
        if len(gb):
            cand = np.where(~used, ious[k], -1)
            j = int(cand.argmax())
            if cand[j] >= IOU_MATCH:
                tp[k], used[j] = True, True
    return tp, used, pc[order], np.where(pm)[0][order], np.where(gm)[0]

def ap50(recs, cls):
    scores, tps, n_gt = [], [], 0
    for r in recs:
        tp, used, sc, _, _ = match_image(r, cls, SWEEP_CONF)
        scores += list(sc); tps += list(tp); n_gt += len(used)
    if n_gt == 0:
        return float('nan')
    o = np.argsort(-np.array(scores))
    tp = np.array(tps, float)[o]
    rec, prec = np.cumsum(tp) / n_gt, np.cumsum(tp) / np.arange(1, len(tp) + 1)
    mrec, mpre = np.r_[0, rec, 1], np.r_[1, prec, 0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    return float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))

def prf(recs, cls, thr):
    tp = fp = fn = 0
    for r in recs:
        t, used, *_ = match_image(r, cls, thr)
        tp += t.sum(); fp += (~t).sum(); fn += (~used).sum()
    p, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return p, rc, 2 * p * rc / max(p + rc, 1e-9)

## 2 · Baseline: old model on the new Go2 test split

The old model only knows `extinguisher`, so its `blue_box` row is empty by construction. The extinguisher row is the honest "before" number: its 0.987 mAP50 was measured on Roboflow's own validation images, not on robot footage.

In [ ]:
baseline = None
if OLD_MODEL and Path(OLD_MODEL).exists():
    old = YOLO(str(OLD_MODEL))
    old_map = {i: CLASSES.index('extinguisher') for i, n in old.names.items() if 'extinguisher' in n.lower()}
    recs_old = collect(old, 'test', old_map)
    baseline = pd.DataFrame([dict(cls=c, AP50=ap50(recs_old, i), **dict(zip(['P', 'R', 'F1'], prf(recs_old, i, 0.5))))
                             for i, c in enumerate(CLASSES)]).set_index('cls').round(3)
    print('old model, test split, conf 0.5 for P/R/F1:')
    display(baseline)
    del old
else:
    print('no baseline model, skipping')

## 3 · Train

Ultralytics transfers every weight whose shape matches, so starting from the old 1-class model keeps the backbone, neck and box regression it learned and re-initialises only the class logits. Set `COMPARE_WITH_COCO_INIT = True` to also train from plain COCO `yolov8n.pt`. The run with the better val mAP50-95 is kept.

In [ ]:
def train(init, name):
    m = YOLO(str(init))
    m.train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
            device=DEVICE, seed=SEED, cache=CACHE, project=str(RUNS_DIR), name=name, exist_ok=False,
            plots=True, **TRAIN_HYP)
    return Path(m.trainer.save_dir)

runs = {'from_old_model': train(INIT_WEIGHTS, RUN_NAME)}
if COMPARE_WITH_COCO_INIT:
    runs['from_coco'] = train('yolov8n.pt', RUN_NAME + '_coco')
runs

In [ ]:
def run_summary(d):
    df = pd.read_csv(d / 'results.csv')
    df.columns = df.columns.str.strip()
    b = df['metrics/mAP50-95(B)'].idxmax()
    return df, dict(run=d.name, best_epoch=int(df.epoch[b]), epochs_run=len(df),
                    val_mAP50=df['metrics/mAP50(B)'][b], val_mAP50_95=df['metrics/mAP50-95(B)'][b])

summ = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for tag, d in runs.items():
    df, s = run_summary(d)
    summ[tag] = s
    for k in ['box', 'cls']:
        axes[0].plot(df.epoch, df[f'train/{k}_loss'], label=f'{tag} train {k}')
        axes[0].plot(df.epoch, df[f'val/{k}_loss'], '--', label=f'{tag} val {k}')
    axes[1].plot(df.epoch, df['metrics/mAP50(B)'], label=f'{tag} mAP50')
    axes[1].plot(df.epoch, df['metrics/mAP50-95(B)'], label=f'{tag} mAP50-95')
    axes[2].plot(df.epoch, df['metrics/precision(B)'], label=f'{tag} P')
    axes[2].plot(df.epoch, df['metrics/recall(B)'], label=f'{tag} R')
for ax, t in zip(axes, ['losses', 'val mAP', 'val P / R']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

display(pd.DataFrame(summ).T)
BEST_TAG = max(summ, key=lambda t: summ[t]['val_mAP50_95'])
RUN_DIR = runs[BEST_TAG]
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
print('selected:', BEST_TAG, '->', BEST_PT)

## 4 · Evaluate on the test split

The test split is never used for training, early stopping or model selection, so these are the numbers to trust.

In [ ]:
model = YOLO(str(BEST_PT))
val_dir_name = f'{RUN_DIR.name}_test'
m = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
              project=str(RUNS_DIR), name=val_dir_name, exist_ok=True, plots=True, verbose=False)
idx = list(m.box.ap_class_index)
official = pd.DataFrame([dict(cls=CLASSES[c], P=m.box.p[k], R=m.box.r[k], mAP50=m.box.ap50[k], mAP50_95=m.box.ap[k])
                         for k, c in enumerate(idx)]).set_index('cls').round(3)
official.loc['all'] = [m.box.mp, m.box.mr, m.box.map50, m.box.map]
display(official.round(3))
VAL_DIR = Path(m.save_dir)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, f in zip(axes, ['confusion_matrix_normalized.png', 'BoxPR_curve.png']):
    p = VAL_DIR / f
    if p.exists():
        ax.imshow(plt.imread(p)); ax.set_title(f)
    ax.axis('off')
plt.tight_layout(); plt.show()

### Confidence threshold for the ROS node

`yolo_subscriber_node.py` uses **one** `conf_thres` for all classes. Recall matters most for finding targets, and false positives also create wrong `/yolo/target_pose` goals. The default choice below maximises the mean F1 over classes.

In [ ]:
recs = collect(model, 'test')
thrs = np.round(np.arange(0.05, 0.951, 0.05), 2)
sweep = pd.DataFrame([dict(thr=t, cls=c, **dict(zip(['P', 'R', 'F1'], prf(recs, i, t))))
                      for t in thrs for i, c in enumerate(CLASSES)])
mean_f1 = sweep.groupby('thr').F1.mean()
CONF_THRES = float(mean_f1.idxmax())

fig, axes = plt.subplots(1, len(CLASSES), figsize=(6 * len(CLASSES), 3.8), squeeze=False)
for ax, c in zip(axes[0], CLASSES):
    s = sweep[sweep.cls == c]
    for k in ['P', 'R', 'F1']:
        ax.plot(s.thr, s[k], label=k)
    ax.axvline(CONF_THRES, ls='--', c='k'); ax.set_title(c); ax.set_xlabel('conf_thres'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'recommended CONF_THRES = {CONF_THRES} (mean F1 {mean_f1.max():.3f})')
display(sweep[sweep.thr.isin([0.25, CONF_THRES, 0.5])].pivot(index='thr', columns='cls').round(3))

new = pd.DataFrame([dict(cls=c, AP50=ap50(recs, i), **dict(zip(['P', 'R', 'F1'], prf(recs, i, 0.5))))
                    for i, c in enumerate(CLASSES)]).set_index('cls').round(3)
if baseline is not None:
    print('old vs new model on the same test split (AP50; P/R/F1 at conf 0.5):')
    display(pd.concat({'old': baseline, 'new': new}, axis=1))

### Recall by object size (a proxy for distance)

COCO size bins on `sqrt(box area)` in pixels: small < 32 ≤ medium < 96 ≤ large. Low recall on *small* means the robot only detects targets when it is close. The fix is more far-away examples in the data, or a larger `IMGSZ` (slower on the Jetson).

In [ ]:
bins = [('small', 0, 32), ('medium', 32, 96), ('large', 96, 1e9)]
rows = []
for i, c in enumerate(CLASSES):
    hit = {b: [0, 0] for b, *_ in bins}
    for r in recs:
        _, used, _, _, gidx = match_image(r, i, CONF_THRES)
        for u, g in zip(used, r['gt_xyxy'][gidx]):
            s = math.sqrt(max((g[2] - g[0]) * (g[3] - g[1]), 0))
            b = next(n for n, lo, hi in bins if lo <= s < hi)
            hit[b][0] += u; hit[b][1] += 1
    rows.append({'cls': c, **{f'{b} recall': (h[0] / h[1] if h[1] else np.nan) for b, h in hit.items()},
                 **{f'{b} n': h[1] for b, h in hit.items()}})
display(pd.DataFrame(rows).set_index('cls').round(3))

### Error gallery at the chosen threshold

Green = ground truth, blue = correct detection, red = **false positive**, yellow = **missed** (false negative).
Look for patterns: one location, lighting, distance, a specific look-alike object. These drive the next labeling round.

In [ ]:
def render(r, thr):
    im = cv2.imread(r['file'])
    nfp = nfn = 0
    for i, c in enumerate(CLASSES):
        tp, used, sc, pidx, gidx = match_image(r, i, thr)
        for g, u in zip(r['gt_xyxy'][gidx].astype(int), used):
            cv2.rectangle(im, tuple(g[:2]), tuple(g[2:]), (0, 200, 0) if u else (0, 220, 255), 2 if u else 4)
            nfn += not u
        for p, t, s in zip(r['p_xyxy'][pidx].astype(int), tp, sc):
            col = (255, 120, 0) if t else (0, 0, 255)
            cv2.rectangle(im, tuple(p[:2]), tuple(p[2:]), col, 2)
            cv2.putText(im, f'{c} {s:.2f}', (p[0], max(12, p[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
            nfp += not t
    return im, nfp, nfn

errs = []
for r in recs:
    im, nfp, nfn = render(r, CONF_THRES)
    if nfp or nfn:
        errs.append((im, f'{Path(r["file"]).name}  FP={nfp} FN={nfn}'))
print(f'{len(errs)} / {len(recs)} test images contain at least one error')
show = random.sample(errs, min(12, len(errs)))
if show:
    n = math.ceil(len(show) / 3)
    fig, axes = plt.subplots(n, 3, figsize=(18, 4.2 * n), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, (im, cap) in zip(axes.flat, show):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(cap, fontsize=9)
    plt.tight_layout(); plt.show()

## 5 · Export

* The `.pt` is copied to `models/<EXPORT_NAME>.pt`, which `run_yolo.sh` mounts into the container.
* ONNX is exported here as a portable artefact and to check that the graph exports cleanly. Outputs are compared against the `.pt`.
* **The TensorRT `.engine` must be built on the Jetson itself**: engines are tied to the GPU and TensorRT version, so one built on this workstation will not load on the Orin Nano. Commands are printed below.

In [ ]:
dst_pt = ROOT / 'models' / f'{EXPORT_NAME}.pt'
shutil.copy2(BEST_PT, dst_pt)
print('weights ->', dst_pt, f'({dst_pt.stat().st_size / 1e6:.1f} MB)')

onnx_path = Path(YOLO(str(BEST_PT)).export(format='onnx', imgsz=IMGSZ, simplify=True))
print('onnx ->', onnx_path)

# Sanity check: ONNX vs PyTorch detections on a few test images
onnx_model, pt_model = YOLO(str(onnx_path), task='detect'), YOLO(str(BEST_PT))
diffs = []
for p in random.sample(split_files('test'), min(10, len(split_files('test')))):
    a = pt_model.predict(str(p), conf=CONF_THRES, imgsz=IMGSZ, verbose=False, device=DEVICE)[0].boxes
    b = onnx_model.predict(str(p), conf=CONF_THRES, imgsz=IMGSZ, verbose=False)[0].boxes
    diffs.append(len(a) - len(b))
print('PyTorch minus ONNX detection count per image:', diffs, '(expect all 0, occasionally +/-1 at the threshold edge)')

In [ ]:
card = {
    'name': EXPORT_NAME,
    'date': str(date.today()),
    'classes': {i: c for i, c in enumerate(CLASSES)},
    'architecture': Path(str(INIT_WEIGHTS if BEST_TAG == 'from_old_model' else 'yolov8n.pt')).name + ' -> fine-tuned',
    'ultralytics': ultralytics.__version__,
    'imgsz': IMGSZ,
    'run_dir': str(RUN_DIR),
    'train_summary': {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in summ[BEST_TAG].items()},
    'test_metrics': official.reset_index().to_dict(orient='records'),
    'recommended_conf_thres': CONF_THRES,
    'dataset': str(DATA_YAML),
}
card_path = ROOT / 'models' / f'{EXPORT_NAME}.json'
card_path.write_text(json.dumps(card, indent=2, default=float))
print(card_path.read_text())

In [ ]:
print(f"""
# ---- 1. copy weights to the Go2 (adjust user/host/path) ----
scp {dst_pt.relative_to(ROOT)} <go2-user>@<go2-host>:<path-to>/GO2_YOLO_THDTCC/models/

# ---- 2. on the Jetson, inside GO2_YOLO_THDTCC/: build the TensorRT engine (FP16), once ----
docker run --rm --runtime nvidia -v "$PWD/models:/models" --entrypoint python3 go2-yolo:latest \\
  -c "from ultralytics import YOLO; YOLO('/models/{EXPORT_NAME}.pt').export(format='engine', half=True, imgsz={IMGSZ})"

# ---- 3. run the node with the new model ----
HOST_MODEL_PATH=$PWD/models/{EXPORT_NAME}.engine CONF_THRES={CONF_THRES} ./run_yolo.sh
#   (or the .pt directly, slower:  HOST_MODEL_PATH=$PWD/models/{EXPORT_NAME}.pt ...)
#   CLASSES_FILTER=0 -> extinguisher only, CLASSES_FILTER=1 -> blue_box only
""")

**Note on `/yolo/target_pose`:** the node publishes the pose of the *single most confident* detection among the classes that pass `CLASSES_FILTER`. With both classes enabled, the target can switch between an extinguisher and a blue box from frame to frame. Use `CLASSES_FILTER` to choose which class the robot should go to, or extend the node to publish one pose topic per class.